# VIMA3YA — NimbusPay ticket triage (Kaggle T4 / Colab T4)

**Training procedure:** Qwen2.5-1.5B-Instruct with PEFT LoRA or 4-bit QLoRA. This notebook runs five controlled trainings, evaluates with the provided scorer, exports the highest-dev-exact-match adapter and model-generated test predictions.

**Reproducibility:** The single GPU `cuda:0` is used, even on Kaggle T4×2. The inference path preserves the original system/user messages, greedy decoding (`do_sample=False`), `max_new_tokens=200`, and strips whitespace only. No field repairs at inference.

**Before starting:** Add the provided assignment files as a Kaggle Dataset input (`train.jsonl`, `dev.jsonl`, `test_inputs.jsonl`, `SCHEMA.md`, `score.py`) or upload to `/kaggle/working`. Turn Internet on for the first model download. On Colab, upload files under `/content`.

In [ ]:

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

print("Visible GPUs:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

import os

files_to_check = [
    "train_clean.jsonl",
    "cleaning_log.csv",
    "data_audit.md",
    "baseline_zero_shot_metrics.json",
    "baseline_2_metrics.json"
]

for filename in files_to_check:
    path = "/kaggle/working/" + filename
    print(filename, ":", "FOUND" if os.path.exists(path) else "MISSING")


In [ ]:

import json

clean_path = "/kaggle/working/train_clean.jsonl"

with open(clean_path, "r", encoding="utf-8") as f:
    clean = [json.loads(line) for line in f if line.strip()]

print("Cleaned training examples:", len(clean))
print("First example ID:", clean[0]["id"])
print("GPU count:", torch.cuda.device_count())

assert len(clean) == 1420
assert torch.cuda.device_count() == 1

print("Dataset restored successfully!")


In [ ]:

import sys
import time
import random
import pathlib
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BASE_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUT = pathlib.Path("/kaggle/working")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Model:", BASE_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("Visible GPUs:", torch.cuda.device_count())
print("Imports and configuration restored!")


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(
    BASE_ID,
    trust_remote_code=False
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16,
    trust_remote_code=False
)

print("Model loaded successfully!")
print("Model:", BASE_ID)
print("Device:", next(model.parameters()).device)
print("Visible GPUs:", torch.cuda.device_count())


In [ ]:

def prompt_ids(messages):
    return tokenizer.apply_chat_template(
        messages[:2], tokenize=True, add_generation_prompt=True
    )

def full_ids(messages):
    return tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=False
    )

MAX_LENGTH = 768

def encode_example(row):
    msgs = row["messages"]
    inp = full_ids(msgs)
    prefix = prompt_ids(msgs)

    if inp[:len(prefix)] != prefix:
        raise ValueError("Chat template prefix mismatch")

    labels = [-100] * len(prefix) + inp[len(prefix):]

    if len(inp) > MAX_LENGTH:
        raise ValueError("Training example exceeds MAX_LENGTH")

    return {"input_ids": inp, "labels": labels}

train_ready = [
    row for row in clean
    if len(full_ids(row["messages"])) <= MAX_LENGTH
]

from torch.utils.data import Dataset

class TriageDataset(Dataset):
    def __init__(self, rows):
        self.examples = [encode_example(row) for row in rows]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]

train_dataset = TriageDataset(train_ready)

print("Training examples:", len(train_dataset))
print("MAX_LENGTH:", MAX_LENGTH)
print("Dataset restored successfully!")

assert len(train_dataset) == 1380


In [ ]:

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", f"{trainable_params:,}")
print("Visible GPUs:", torch.cuda.device_count())

assert trainable_params < 50_000_000
assert torch.cuda.device_count() == 1

print("LoRA adapter restored successfully!")


In [ ]:

def collate(examples):
    pad = tokenizer.pad_token_id
    m = max(len(x["input_ids"]) for x in examples)

    ids = [
        x["input_ids"] + [pad] * (m - len(x["input_ids"]))
        for x in examples
    ]

    att = [
        [1] * len(x["input_ids"]) +
        [0] * (m - len(x["input_ids"]))
        for x in examples
    ]

    lbs = [
        x["labels"] + [-100] * (m - len(x["labels"]))
        for x in examples
    ]

    return {
        "input_ids": torch.tensor(ids, dtype=torch.long),
        "attention_mask": torch.tensor(att, dtype=torch.long),
        "labels": torch.tensor(lbs, dtype=torch.long)
    }

sample_batch = collate([train_dataset[0], train_dataset[1]])

print("Batch device:", sample_batch["input_ids"].device)
print("Batch shape:", sample_batch["input_ids"].shape)
print("Collator restored successfully!")

assert sample_batch["input_ids"].device.type == "cpu"


In [ ]:

training_args = TrainingArguments(
    output_dir=str(OUT / "triage_lora_checkpoints"),
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    seed=42,
    data_seed=42,
    remove_unused_columns=False,
    dataloader_num_workers=0
)

print("Visible GPUs:", torch.cuda.device_count())
print("Trainer GPUs:", training_args.n_gpu)
print("Epochs:", training_args.num_train_epochs)
print("Effective batch size:",
      training_args.per_device_train_batch_size *
      training_args.gradient_accumulation_steps)

assert torch.cuda.device_count() == 1
assert training_args.n_gpu == 1

print("Training settings restored successfully!")


In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate
)

print("Trainer initialized successfully!")
print("Training examples:", len(trainer.train_dataset))
print("Trainer GPUs:", trainer.args.n_gpu)
print("Model device:", next(trainer.model.parameters()).device)

assert trainer.args.n_gpu == 1

print("Ready to start single-GPU QLoRA training!")


In [ ]:

import time

training_start = time.time()

train_result = trainer.train()

training_minutes = (time.time() - training_start) / 60

print("Training completed successfully!")
print("Training time (minutes):", round(training_minutes, 2))
print("Final training loss:", train_result.training_loss)
print("Global steps:", trainer.state.global_step)


In [ ]:

from pathlib import Path

adapter_dir = Path("/kaggle/working/adapter")
adapter_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

print("LoRA adapter saved successfully!")
print("Saved location:", adapter_dir)

for file in sorted(adapter_dir.iterdir()):
    print(file.name, "-", round(file.stat().st_size / (1024 * 1024), 2), "MB")

assert (adapter_dir / "adapter_config.json").exists()
assert any(adapter_dir.glob("adapter_model.*"))

print("Adapter files verified successfully!")


In [ ]:

import shutil
from pathlib import Path

backup_path = shutil.make_archive(
    "/kaggle/working/adapter_backup",
    "zip",
    root_dir="/kaggle/working",
    base_dir="adapter"
)

print("Backup created:", backup_path)
print("Backup size:",
      round(Path(backup_path).stat().st_size / (1024 * 1024), 2),
      "MB")

assert Path(backup_path).exists()
print("Backup verified successfully!")


In [ ]:

import json
from pathlib import Path

dev_path = Path(
    "/kaggle/input/datasets/keerthiyp/assignment/dev.jsonl"
)

with open(dev_path, "r", encoding="utf-8") as f:
    dev_rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Development examples:", len(dev_rows))
print("First example ID:", dev_rows[0].get("id"))
print("Available keys:", list(dev_rows[0].keys()))
print("Model ready:", model is not None)
print("Tokenizer ready:", tokenizer is not None)


In [ ]:
import torch
import time

model.eval()

sample = dev_rows[0]

# Use only the original system and user messages.
messages = sample["messages"][:2]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda:0")

start = time.time()

with torch.inference_mode():
    generated = model.generate(
        input_ids=inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

# Decode only the newly generated tokens.
new_tokens = generated[0][inputs.shape[1]:]

raw_output = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("Example ID:", sample["id"])
print("Generated output:")
print(raw_output)
print("Generation time:", round(time.time() - start, 2), "seconds")

try:
    parsed = json.loads(raw_output)
    print("Valid JSON: YES")
    print("Output fields:", list(parsed.keys()))
except json.JSONDecodeError:
    print("Valid JSON: NO")



In [ ]:
!pip -q install "transformers>=4.48,<5" "peft>=0.14,<1" "accelerate>=1.2,<2" "bitsandbytes>=0.45,<1" "safetensors>=0.4" "python-dateutil>=2.8" "sentencepiece"

In [ ]:

import transformers
import peft
import accelerate
import bitsandbytes
import torch

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())


In [ ]:

import torch

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")
print("Number of GPUs:", torch.cuda.device_count())


In [ ]:

import torch
import time

model.eval()

sample = dev_rows[0]

# Use only the original system and user messages.
messages = sample["messages"][:2]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda:0")

start = time.time()

with torch.inference_mode():
    generated = model.generate(
        input_ids=inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

# Decode only the newly generated tokens.
new_tokens = generated[0][inputs.shape[1]:]

raw_output = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("Example ID:", sample["id"])
print("Generated output:")
print(raw_output)
print("Generation time:", round(time.time() - start, 2), "seconds")

try:
    parsed = json.loads(raw_output)
    print("Valid JSON: YES")
    print("Output fields:", list(parsed.keys()))
except json.JSONDecodeError:
    print("Valid JSON: NO")


In [ ]:

import json
import time
import torch
from pathlib import Path

model.eval()
dev_predictions = []
start = time.time()

for i, row in enumerate(dev_rows):
    inputs = tokenizer.apply_chat_template(
        row["messages"][:2],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda:0")

    with torch.inference_mode():
        generated = model.generate(
            input_ids=inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    output = tokenizer.decode(
        generated[0][inputs.shape[1]:],
        skip_special_tokens=True
    ).strip()

    dev_predictions.append({
        "id": row["id"],
        "output": output
    })

    if (i + 1) % 25 == 0:
        print(f"Completed {i + 1}/{len(dev_rows)}")

output_path = Path(
    "/kaggle/working/qlora_dev_predictions.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for prediction in dev_predictions:
        f.write(json.dumps(prediction, ensure_ascii=False) + "\n")

valid_count = 0

for prediction in dev_predictions:
    try:
        parsed = json.loads(prediction["output"])
        if isinstance(parsed, dict):
            valid_count += 1
    except json.JSONDecodeError:
        pass

print("\nEvaluation generation completed!")
print("Total predictions:", len(dev_predictions))
print("Valid JSON:", valid_count)
print("JSON validity:", round(100 * valid_count / len(dev_predictions), 2), "%")
print("Time (minutes):", round((time.time() - start) / 60, 2))
print("Saved:", output_path)


In [ ]:

from pathlib import Path

score_path = Path(
    "/kaggle/input/datasets/keerthiyp/assignment/score.py"
)

print("Scoring script exists:", score_path.exists())
print("\n===== SCORING SCRIPT =====\n")
print(score_path.read_text(encoding="utf-8"))


In [ ]:

import subprocess
import json
from pathlib import Path

score_script = "/kaggle/input/datasets/keerthiyp/assignment/score.py"
gold_file = "/kaggle/input/datasets/keerthiyp/assignment/dev.jsonl"
pred_file = "/kaggle/working/qlora_dev_predictions.jsonl"

result = subprocess.run(
    [
        "python", score_script,
        "--gold", gold_file,
        "--pred", pred_file,
        "--json"
    ],
    capture_output=True,
    text=True,
    check=True
)

scores = json.loads(result.stdout)

metrics_path = Path("/kaggle/working/qlora_dev_metrics.json")
metrics_path.write_text(
    json.dumps(scores, indent=2),
    encoding="utf-8"
)

print("===== OFFICIAL DEV RESULTS =====")
print("JSON validity:", f"{scores['all']['json_valid']:.2%}")
print("Exact match:", f"{scores['all']['exact_match']:.2%}")
print("Mean field accuracy:", f"{scores['all']['mean_field_acc']:.2%}")

print("\n===== PER-FIELD ACCURACY =====")
for field, accuracy in scores["all"]["fields"].items():
    print(f"{field:15}: {accuracy:.2%}")

print("\nMetrics saved to:", metrics_path)


In [ ]:

from pathlib import Path

schema_path = Path(
    "/kaggle/input/datasets/keerthiyp/assignment/SCHEMA.md"
)

print("===== SCHEMA RULES =====")
print(schema_path.read_text(encoding="utf-8"))


In [ ]:

from pathlib import Path

schema_text = Path(
    "/kaggle/input/datasets/keerthiyp/assignment/SCHEMA.md"
).read_text(encoding="utf-8")

RULES_PROMPT = (
    "Follow these NimbusPay triage rules when producing "
    "the requested JSON output.\n\n"
    + schema_text
    + "\n\nReturn exactly one JSON object with the eight "
      "required keys. No Markdown or explanations."
)

print("Rules prompt prepared successfully!")
print("Prompt length:", len(RULES_PROMPT), "characters")
print("Schema included:", "needs_human" in RULES_PROMPT)


In [ ]:

import gc
import torch
from transformers import AutoModelForCausalLM

# Release the trained model from GPU memory.
del trainer
del model

gc.collect()
torch.cuda.empty_cache()

# Load the original base model without LoRA.
baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16,
    trust_remote_code=False
)

baseline_model.eval()

print("Original base model loaded successfully!")
print("Model:", BASE_ID)
print("Device:", next(baseline_model.parameters()).device)
print("LoRA adapter attached:", hasattr(baseline_model, "peft_config"))


In [ ]:

import json
import time
import torch

sample = dev_rows[0]

# Preserve the original user message.
# Add schema rules to the original system message.
messages = [
    {
        "role": "system",
        "content": (
            sample["messages"][0]["content"]
            + "\n\n"
            + RULES_PROMPT
        )
    },
    sample["messages"][1]
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda:0")

start = time.time()

with torch.inference_mode():
    generated = baseline_model.generate(
        input_ids=inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

raw_output = tokenizer.decode(
    generated[0][inputs.shape[1]:],
    skip_special_tokens=True
).strip()

print("Example ID:", sample["id"])
print("Generated output:")
print(raw_output)
print("Generation time:", round(time.time() - start, 2), "seconds")

try:
    parsed = json.loads(raw_output)
    print("Valid JSON:", isinstance(parsed, dict))
    if isinstance(parsed, dict):
        print("Output fields:", list(parsed.keys()))
except json.JSONDecodeError:
    print("Valid JSON: False")


In [ ]:

import json
import time
import torch
from pathlib import Path

baseline_model.eval()
rules_predictions = []
start = time.time()

output_path = Path(
    "/kaggle/working/baseline_rules_predictions.jsonl"
)

for i, row in enumerate(dev_rows):
    messages = [
        {
            "role": "system",
            "content": (
                row["messages"][0]["content"]
                + "\n\n"
                + RULES_PROMPT
            )
        },
        row["messages"][1]
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda:0")

    with torch.inference_mode():
        generated = baseline_model.generate(
            input_ids=inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    raw_output = tokenizer.decode(
        generated[0][inputs.shape[1]:],
        skip_special_tokens=True
    ).strip()

    rules_predictions.append({
        "id": row["id"],
        "output": raw_output
    })

    # Save progress every 25 examples.
    if (i + 1) % 25 == 0:
        with open(output_path, "w", encoding="utf-8") as f:
            for prediction in rules_predictions:
                f.write(json.dumps(prediction, ensure_ascii=False) + "\n")

        print(f"Completed {i + 1}/{len(dev_rows)}")

print("\nRules-based baseline inference completed!")
print("Predictions:", len(rules_predictions))
print("Time (minutes):", round((time.time() - start) / 60, 2))
print("Saved:", output_path)


In [ ]:

import subprocess
import json
from pathlib import Path

result = subprocess.run(
    [
        "python",
        "/kaggle/input/datasets/keerthiyp/assignment/score.py",
        "--gold",
        "/kaggle/input/datasets/keerthiyp/assignment/dev.jsonl",
        "--pred",
        "/kaggle/working/baseline_rules_predictions.jsonl",
        "--json"
    ],
    capture_output=True,
    text=True,
    check=True
)

rules_scores = json.loads(result.stdout)

metrics_path = Path(
    "/kaggle/working/baseline_rules_metrics.json"
)

metrics_path.write_text(
    json.dumps(rules_scores, indent=2),
    encoding="utf-8"
)

print("===== RULES-BASED BASELINE =====")
print("JSON validity:", f"{rules_scores['all']['json_valid']:.2%}")
print("Exact match:", f"{rules_scores['all']['exact_match']:.2%}")
print("Mean field accuracy:", f"{rules_scores['all']['mean_field_acc']:.2%}")

print("\n===== TRAINED QLORA MODEL =====")
print("JSON validity:", f"{scores['all']['json_valid']:.2%}")
print("Exact match:", f"{scores['all']['exact_match']:.2%}")
print("Mean field accuracy:", f"{scores['all']['mean_field_acc']:.2%}")

print("\nMetrics saved to:", metrics_path)


In [ ]:

from pathlib import Path

files = [
    "/kaggle/working/baseline_rules_predictions.jsonl",
    "/kaggle/working/baseline_rules_metrics.json"
]

for file in files:
    p = Path(file)
    print(p.name)
    print("Exists:", p.exists())
    if p.exists():
        print("Size:", round(p.stat().st_size / 1024, 2), "KB")
    print()


In [ ]:

import zipfile
from pathlib import Path

backup = Path("/kaggle/working/part_b_backup.zip")

files = [
    Path("/kaggle/working/baseline_rules_predictions.jsonl"),
    Path("/kaggle/working/baseline_rules_metrics.json")
]

with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files:
        z.write(file, arcname=file.name)

print("Backup created:", backup)
print("Size:", round(backup.stat().st_size / 1024, 2), "KB")

with zipfile.ZipFile(backup) as z:
    print("Files inside ZIP:", z.namelist())

print("Part B backup ready!")


In [ ]:

from IPython.display import display, FileLink
from pathlib import Path

file_path = Path("/kaggle/working/part_b_backup.zip")

if file_path.exists():
    print("Backup found!")
    display(FileLink(str(file_path)))
else:
    print("Backup file not found.")


In [ ]:

from IPython.display import FileLink, display
from pathlib import Path
import shutil

source = Path("/kaggle/working/part_b_backup.zip")
destination = Path("/kaggle/working/part_b_download.zip")

assert source.exists(), "Backup file missing!"

shutil.copy2(source, destination)

print("File ready:", destination.name)
print("Size:", destination.stat().st_size, "bytes")
print("Download link:")
display(FileLink(destination.name))


In [ ]:

from pathlib import Path
import torch

working = Path("/kaggle/working")

resources = {
    "Cleaned training data": working / "train_clean.jsonl",
    "Raw training data": Path(
        "/kaggle/input/datasets/keerthiyp/assignment/train.jsonl"
    ),
    "Trained LoRA adapter": working / "adapter",
    "Training checkpoints": working / "triage_lora_checkpoints",
    "Dev predictions": working / "qlora_dev_predictions.jsonl",
    "Dev metrics": working / "qlora_dev_metrics.json"
}

for name, path in resources.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

    if path.is_dir():
        print("  Contents:", [p.name for p in path.iterdir()][:10])

print("\nGPU available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())


In [ ]:

import json
from pathlib import Path

raw_path = Path(
    "/kaggle/input/datasets/keerthiyp/assignment/train.jsonl"
)

with open(raw_path, encoding="utf-8") as f:
    raw_rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

valid_labels = 0
invalid_labels = 0
within_limit = 0

for row in raw_rows:
    try:
        json.loads(row["messages"][-1]["content"])
        valid_labels += 1

        if len(full_ids(row["messages"])) <= MAX_LENGTH:
            within_limit += 1

    except (json.JSONDecodeError, KeyError, TypeError):
        invalid_labels += 1

print("Total raw records:", len(raw_rows))
print("Valid JSON labels:", valid_labels)
print("Invalid JSON labels:", invalid_labels)
print("Valid labels within token limit:", within_limit)
print("MAX_LENGTH:", MAX_LENGTH)


In [ ]:

# Prepare raw dataset without applying label corrections.

raw_train_ready = []

for row in raw_rows:
    try:
        json.loads(row["messages"][-1]["content"])

        if len(full_ids(row["messages"])) <= MAX_LENGTH:
            raw_train_ready.append(row)

    except (json.JSONDecodeError, KeyError, TypeError):
        continue

raw_train_dataset = TriageDataset(raw_train_ready)

print("Raw training examples:", len(raw_train_dataset))
print("Cleaned training examples:", len(train_dataset))

raw_ids = {row["id"] for row in raw_train_ready}
clean_ids = {row["id"] for row in train_ready}

print("Same training IDs:", raw_ids == clean_ids)

assert len(raw_train_dataset) == 1380
assert raw_ids == clean_ids

print("Raw dataset prepared successfully!")


In [ ]:

import gc
import torch
from transformers import AutoModelForCausalLM

# Release the baseline model from GPU memory.
del baseline_model

gc.collect()
torch.cuda.empty_cache()

# Load a fresh 4-bit base model.
raw_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16,
    trust_remote_code=False
)

raw_model = prepare_model_for_kbit_training(raw_model)
raw_model = get_peft_model(raw_model, lora_config)

trainable = sum(
    p.numel()
    for p in raw_model.parameters()
    if p.requires_grad
)

print("Fresh raw-data model loaded!")
print("Device:", next(raw_model.parameters()).device)
print("Trainable parameters:", f"{trainable:,}")
print("Visible GPUs:", torch.cuda.device_count())

assert trainable == 18_464_768
assert torch.cuda.device_count() == 1

print("Ready to configure Experiment 1 training!")


In [ ]:

from transformers import TrainingArguments, Trainer

raw_training_args = TrainingArguments(
    output_dir="/kaggle/working/experiment1_raw_checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    seed=42,
    data_seed=42,
    remove_unused_columns=False,
    dataloader_num_workers=0
)

raw_trainer = Trainer(
    model=raw_model,
    args=raw_training_args,
    train_dataset=raw_train_dataset,
    data_collator=collate
)

print("Experiment 1 Trainer initialized!")
print("Training examples:", len(raw_trainer.train_dataset))
print("Epochs:", raw_training_args.num_train_epochs)
print("Trainer GPUs:", raw_training_args.n_gpu)
print("Checkpoint directory:", raw_training_args.output_dir)

assert raw_training_args.n_gpu == 1

print("Ready for raw-data training!")


In [ ]:

import time
import torch

torch.cuda.reset_peak_memory_stats()
start = time.time()

raw_train_result = raw_trainer.train()

raw_training_minutes = (time.time() - start) / 60
raw_peak_gpu_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

print("\n===== EXPERIMENT 1: RAW DATA =====")
print("Training completed successfully!")
print("Training time:", round(raw_training_minutes, 2), "minutes")
print("Final training loss:", raw_train_result.training_loss)
print("Global steps:", raw_trainer.state.global_step)
print("Peak GPU memory:", round(raw_peak_gpu_gb, 2), "GB")


## A. Train-only label audit and repair

Schema checks include required fields, valid dates, normalized `NP` transaction IDs, category-specific null fields, priority thresholds and legal threats. Invalid JSON labels are dropped, not reconstructed. All fixes and drops are recorded by ticket ID.

**Limits of the repair:** Keyword detection for legal threats cannot capture every paraphrase; ambiguous transaction/date semantics need manual audit. We do not use development or test labels to modify training rows.

In [ ]:
KEYS=['category','priority','amount_paise','txn_date','txn_id','channel','language','needs_human']
CATS={'payment_failed','refund','fraud','account_access','kyc','offers','other'}

def clean_one(row):
    r=copy.deepcopy(row); s=r['messages'][-1]['content']; ticket=r['messages'][1]['content']; reasons=[]
    try:d=json.loads(s)
    except (ValueError,TypeError):return None,'drop','invalid assistant JSON'
    if not isinstance(d,dict):return None,'drop','assistant JSON not an object'
    if any(k not in d for k in KEYS if k!='priority'):return None,'drop','missing required keys besides priority'
    if d.get('category') not in CATS or d.get('language') not in ('en','hi-en') or d.get('channel') not in (None,'upi','card','netbanking','wallet') or type(d.get('needs_human')) is not bool or (d.get('amount_paise') is not None and type(d.get('amount_paise')) is not int):return None,'drop','invalid field type or enumeration'
    if d.get('txn_date') is not None:
        v=d['txn_date']
        if not re.fullmatch(r'\d{4}-\d{2}-\d{2}',str(v)):
            try:
                recv=re.search(r'Received: (\d{4}-\d{2}-\d{2})',ticket)
                ref=datetime.date.fromisoformat(recv.group(1)) if recv else datetime.date(2026,1,1)
                parsed=date_parser.parse(v,dayfirst=True,default=datetime.datetime(ref.year,1,1)).date()
                if parsed>ref:parsed=parsed.replace(year=parsed.year-1)
                d['txn_date']=parsed.isoformat(); reasons.append('normalized transaction date')
            except Exception:return None,'drop','ambiguous or invalid date'
        try:datetime.date.fromisoformat(d['txn_date'])
        except:return None,'drop','invalid calendar date'
    if d.get('txn_id') is not None and not re.fullmatch(r'NP\d{10}',str(d['txn_id'])):
        normalized=re.sub(r'[^A-Za-z0-9]','',str(d['txn_id'])).upper()
        if re.fullmatch(r'NP\d{10}',normalized):d['txn_id']=normalized;reasons.append('normalized NimbusPay transaction id')
        else:return None,'drop','invalid NimbusPay transaction id'
    if d['category'] in ('account_access','kyc','other'):
        for k in ('amount_paise','txn_date','txn_id','channel'):
            if d[k] is not None:d[k]=None;reasons.append('cleared transaction fields for nontransaction category')
    a=d['amount_paise']; c=d['category'];
    # Legal keywords from the spec. Ignore quoted earlier email lines, to reduce false positives.
    active='\n'.join(l for l in ticket.splitlines() if not l.lstrip().startswith('>'))
    legal=bool(re.search(r'\bRBI\b|ombudsman|consumer court|lawyer|legal notice|legal action',active,re.I))
    expected='P1' if c=='fraud' or a is not None and a>=5000000 or legal else 'P2' if c=='account_access' or c in ('payment_failed','refund') and a is not None and a>=500000 else 'P3'
    if d.get('priority')!=expected:reasons.append('recomputed priority from schema thresholds and legal keywords');d['priority']=expected
    if d['priority']=='P1' and d['needs_human'] is False:d['needs_human']=True;reasons.append('P1 requires human review')
    # Do not turn true to false: prior support contacts can still mandate a human.
    r['messages'][-1]['content']=json.dumps({k:d[k] for k in KEYS},ensure_ascii=False,separators=(',',':'))
    return r,'fix' if reasons else None,'; '.join(dict.fromkeys(reasons))


clean=[];log=[];issues=collections.Counter()
for row in raw:
    new,action,reason=clean_one(row)
    if action:log.append({'id':row['id'],'action':action,'reason':reason});issues[action]+=1
    if new:clean.append(new)
with (OUT/'cleaning_log.csv').open('w',newline='',encoding='utf8') as f:
    writer=csv.DictWriter(f,fieldnames=['id','action','reason']);writer.writeheader();writer.writerows(log)
with (OUT/'train_clean.jsonl').open('w',encoding='utf8') as f:
    for r in clean:f.write(json.dumps(r,ensure_ascii=False)+'\n')
print('Raw:',len(raw),'Clean:',len(clean),'Actions:',dict(issues))
print('First 12 actions:',log[:12])
assert len(clean)+issues['drop']==len(raw)


In [ ]:

import json
from collections import Counter

categories = Counter()
invalid_json = []
missing_priority = []

for row in raw:
    try:
        label = json.loads(row["messages"][-1]["content"])

        if not isinstance(label, dict):
            invalid_json.append(row["id"])
            continue

    except (ValueError, TypeError):
        invalid_json.append(row["id"])
        continue

    categories[label.get("category")] += 1

    if "priority" not in label:
        missing_priority.append(row["id"])

print("Category distribution:", categories)
print("Invalid JSON count:", len(invalid_json))
print("Missing priority count:", len(missing_priority))
print("Invalid JSON IDs:", invalid_json[:10])
print("Missing priority IDs:", missing_priority[:10])


In [ ]:

for row in raw:
    if row["id"] in missing_priority[:3]:
        print("=" * 60)
        print("ID:", row["id"])
        print("TICKET:")
        print(row["messages"][1]["content"])
        print("ORIGINAL LABEL:")
        print(row["messages"][-1]["content"])
        print()


In [ ]:

import json

check_ids = {"tr-00020", "tr-00049", "tr-00058"}

for row in clean:
    if row["id"] in check_ids:
        label = json.loads(row["messages"][-1]["content"])

        print("ID:", row["id"])
        print(json.dumps(label, indent=2))
        print("-" * 50)


In [ ]:

for row in raw:
    if row["id"] in invalid_json[:5]:
        print("=" * 60)
        print("ID:", row["id"])
        print("TICKET:")
        print(row["messages"][1]["content"])
        print("ORIGINAL LABEL:")
        print(row["messages"][-1]["content"])
        print()


In [ ]:

from collections import Counter

error_types = Counter()

for row in raw:
    if row["id"] in invalid_json:
        label = row["messages"][-1]["content"]

        if not label.rstrip().endswith("}"):
            error_types["truncated_or_missing_closing_brace"] += 1
        else:
            error_types["other_invalid_json"] += 1

print("Invalid JSON breakdown:")
for error, count in error_types.items():
    print(error, ":", count)

print("Total:", sum(error_types.values()))


In [ ]:

from collections import Counter

fix_reasons = Counter()

for entry in log:
    if entry["action"] == "fix":
        for reason in entry["reason"].split("; "):
            fix_reasons[reason] += 1

print("Automatic fix breakdown:")
for reason, count in fix_reasons.most_common():
    print(f"{count:4} | {reason}")

print("\nTotal records modified:", issues["fix"])


In [ ]:

priority_fixes = [
    entry for entry in log
    if entry["action"] == "fix"
    and "recomputed priority" in entry["reason"]
]

for entry in priority_fixes[:5]:
    original = next(r for r in raw if r["id"] == entry["id"])
    corrected = next(r for r in clean if r["id"] == entry["id"])

    old_label = json.loads(original["messages"][-1]["content"])
    new_label = json.loads(corrected["messages"][-1]["content"])

    print("=" * 60)
    print("ID:", entry["id"])
    print("TICKET:")
    print(original["messages"][1]["content"])
    print("OLD PRIORITY:", old_label.get("priority"))
    print("NEW PRIORITY:", new_label.get("priority"))
    print()


In [ ]:

risky_changes = []

for entry in priority_fixes:
    original = next(r for r in raw if r["id"] == entry["id"])
    corrected = next(r for r in clean if r["id"] == entry["id"])

    old = json.loads(original["messages"][-1]["content"])
    new = json.loads(corrected["messages"][-1]["content"])

    if old.get("priority") in ("P2", "P3") and new["priority"] == "P1":
        risky_changes.append({
            "id": entry["id"],
            "old": old["priority"],
            "new": new["priority"],
            "category": new["category"]
        })

print("Existing P2/P3 changed to P1:", len(risky_changes))
print("Examples:")
for item in risky_changes[:10]:
    print(item)


In [ ]:

non_fraud_p1 = [
    item for item in risky_changes
    if item["category"] != "fraud"
]

print("Non-fraud tickets changed to P1:", len(non_fraud_p1))

for item in non_fraud_p1[:5]:
    row = next(r for r in raw if r["id"] == item["id"])

    print("=" * 60)
    print("ID:", item["id"])
    print("CATEGORY:", item["category"])
    print("PRIORITY:", item["old"], "->", item["new"])
    print("TICKET:")
    print(row["messages"][1]["content"])
    print()


In [ ]:

for item in non_fraud_p1[5:]:
    row = next(r for r in raw if r["id"] == item["id"])

    print("=" * 60)
    print("ID:", item["id"])
    print("CATEGORY:", item["category"])
    print("PRIORITY:", item["old"], "->", item["new"])
    print("TICKET:")
    print(row["messages"][1]["content"])
    print()


In [ ]:

from collections import Counter

priority_transitions = Counter()

for entry in priority_fixes:
    original = next(r for r in raw if r["id"] == entry["id"])
    corrected = next(r for r in clean if r["id"] == entry["id"])

    old = json.loads(original["messages"][-1]["content"])
    new = json.loads(corrected["messages"][-1]["content"])

    old_priority = old.get("priority", "MISSING")
    new_priority = new["priority"]

    priority_transitions[(old_priority, new_priority)] += 1

print("Priority correction breakdown:")

for (old, new), count in priority_transitions.most_common():
    print(f"{str(old):>8} -> {new}: {count}")

print("\nTotal:", sum(priority_transitions.values()))


In [ ]:

downgrades = []

for entry in priority_fixes:
    original = next(r for r in raw if r["id"] == entry["id"])
    corrected = next(r for r in clean if r["id"] == entry["id"])

    old = json.loads(original["messages"][-1]["content"])
    new = json.loads(corrected["messages"][-1]["content"])

    if old.get("priority") == "P1" and new["priority"] == "P3":
        downgrades.append(original)

print("P1 -> P3 records:", len(downgrades))

for row in downgrades[:8]:
    print("=" * 60)
    print("ID:", row["id"])
    print("TICKET:")
    print(row["messages"][1]["content"])
    print("ORIGINAL LABEL:")
    print(row["messages"][-1]["content"])
    print()


In [ ]:

for ticket_id in ["tr-00073", "tr-00187"]:
    row = next(r for r in clean if r["id"] == ticket_id)
    label = json.loads(row["messages"][-1]["content"])

    print("ID:", ticket_id)
    print("CLEANED LABEL:")
    print(json.dumps(label, indent=2))
    print("-" * 50)


In [ ]:

from collections import Counter

amount_summary = Counter()

for row in raw:
    try:
        label = json.loads(row["messages"][-1]["content"])
    except (ValueError, TypeError):
        continue

    if not isinstance(label, dict):
        continue

    amount = label.get("amount_paise")

    if isinstance(amount, int) and not isinstance(amount, bool):
        if amount < 500000:
            amount_summary["Below 5 lakh paise"] += 1
        elif amount < 5000000:
            amount_summary["5 lakh to 50 lakh paise"] += 1
        else:
            amount_summary["50 lakh paise or more"] += 1
    else:
        amount_summary["Null or invalid amount"] += 1

print("Original amount-label distribution:")
for group, count in amount_summary.items():
    print(f"{group}: {count}")


In [ ]:

for row in downgrades:
    label = json.loads(row["messages"][-1]["content"])
    ticket = row["messages"][1]["content"]

    print("=" * 60)
    print("ID:", row["id"])
    print("CATEGORY:", label.get("category"))
    print("ORIGINAL AMOUNT_PAISE:", label.get("amount_paise"))
    print("TICKET:")
    print(ticket)


In [ ]:

ticket_id = "tr-00312"

original = next(r for r in raw if r["id"] == ticket_id)
corrected = next(r for r in clean if r["id"] == ticket_id)

old_label = json.loads(original["messages"][-1]["content"])
new_label = json.loads(corrected["messages"][-1]["content"])

print("ID:", ticket_id)
print("Original label:")
print(json.dumps(old_label, indent=2))
print("\nCleaned label:")
print(json.dumps(new_label, indent=2))


In [ ]:

ticket = original["messages"][1]["content"]

quoted_replies = [
    line for line in ticket.splitlines()
    if line.strip().startswith("> On ")
    and "NimbusPay Support wrote:" in line
]

print("Ticket ID:", original["id"])
print("Quoted support replies:", len(quoted_replies))
print("First three reply headers:")

for line in quoted_replies[:3]:
    print(line)


In [ ]:

with open(PATHS["SCHEMA.md"], "r", encoding="utf-8") as f:
    schema_text = f.read()

for line_number, line in enumerate(schema_text.splitlines(), start=1):
    if any(word in line.lower() for word in [
        "human", "repeat", "previous", "contact", "quoted"
    ]):
        print(f"Line {line_number}: {line}")


In [ ]:

lines = schema_text.splitlines()

for i, line in enumerate(lines):
    if "### priority" in line.lower():
        print("\n".join(
            f"{j + 1}: {lines[j]}"
            for j in range(i, min(i + 18, len(lines)))
        ))


In [ ]:

amount_corrections = {
    "tr-00073": 16878725,
    "tr-00187": 22000000
}

for row in clean:
    if row["id"] in amount_corrections:
        label = json.loads(row["messages"][-1]["content"])

        label["amount_paise"] = amount_corrections[row["id"]]
        label["priority"] = "P1"
        label["needs_human"] = True

        row["messages"][-1]["content"] = json.dumps(
            {key: label[key] for key in KEYS},
            ensure_ascii=False,
            separators=(",", ":")
        )

        print(
            row["id"],
            "amount_paise =", label["amount_paise"],
            "priority =", label["priority"],
            "needs_human =", label["needs_human"]
        )


In [ ]:

lakh_suspects = []

for row in clean:
    ticket = row["messages"][1]["content"]
    label = json.loads(row["messages"][-1]["content"])

    if re.search(r"\blakhs?\b", ticket, re.IGNORECASE):
        amount = label.get("amount_paise")

        if amount is not None and amount < 5000000:
            lakh_suspects.append((row["id"], amount, ticket))

print("Potential lakh amount errors:", len(lakh_suspects))

for ticket_id, amount, ticket in lakh_suspects[:10]:
    print("=" * 60)
    print("ID:", ticket_id)
    print("Stored amount_paise:", amount)
    print("TICKET:")
    print(ticket)


In [ ]:

remaining_downgrades = []

for row in downgrades:
    corrected = next(r for r in clean if r["id"] == row["id"])
    label = json.loads(corrected["messages"][-1]["content"])

    if label["priority"] == "P3":
        remaining_downgrades.append(row)

print("Remaining P1 -> P3 records:", len(remaining_downgrades))

for row in remaining_downgrades:
    label = json.loads(row["messages"][-1]["content"])

    print(
        row["id"],
        "| category:", label["category"],
        "| original amount_paise:", label["amount_paise"]
    )


In [ ]:

final_path = OUT / "train_clean.jsonl"

with final_path.open("w", encoding="utf-8") as f:
    for row in clean:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

saved = read_jsonl(final_path)

assert len(saved) == 1420
assert len({row["id"] for row in saved}) == 1420

for row in saved:
    label = json.loads(row["messages"][-1]["content"])
    assert set(label.keys()) == set(KEYS)
    assert label["priority"] in ("P1", "P2", "P3")

for ticket_id, expected_amount in {
    "tr-00073": 16878725,
    "tr-00187": 22000000
}.items():
    row = next(r for r in saved if r["id"] == ticket_id)
    label = json.loads(row["messages"][-1]["content"])
    assert label["amount_paise"] == expected_amount
    assert label["priority"] == "P1"

print("Saved:", final_path)
print("Validated records:", len(saved))
print("Two confirmed amount corrections preserved.")


In [ ]:

audit_report = """# Part A — Training Data Audit

## Dataset
- Original training records: 1,445
- Cleaned training records: 1,420
- Dropped records: 25

## Identified Issues
- 25 invalid JSON labels with missing closing braces.
- 70 records missing the required priority field.
- 30 transaction-date normalizations.
- 20 transaction-ID normalizations.
- 135 priority recalculations.

Note: Correction counts overlap across records.

## Cleaning Decisions
1. Dropped 25 records with invalid JSON labels.
2. Normalized recoverable transaction dates and IDs.
3. Reconstructed missing priorities using SCHEMA.md.
4. Recalculated priorities using schema thresholds.
5. Corrected two confirmed amount-label errors:
   - tr-00073: 16,878,725 paise; P1.
   - tr-00187: 22,000,000 paise; P1.
6. Preserved original ticket messages.

## Limitations
- Not every training label was manually verified.
- Automatic priority corrections depend partly on
  existing amount labels, which may contain errors.
- The cleaned dataset is structurally validated,
  not guaranteed to be semantically error-free.

## Data Separation
- Training cleaning used train.jsonl and SCHEMA.md.
- Development and test records were not used to
  modify training labels.
"""

report_path = OUT / "data_audit.md"
report_path.write_text(audit_report, encoding="utf-8")

print("Audit report saved:", report_path)
print("Cleaned dataset:", OUT / "train_clean.jsonl")


# PART B — BASELINES ****

In [ ]:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    BASE_ID,
    trust_remote_code=False
)

print("Model:", BASE_ID)
print("Tokenizer loaded successfully")
print("Chat template available:", tokenizer.chat_template is not None)
print("Vocabulary size:", len(tokenizer))


In [ ]:

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=quant_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    trust_remote_code=False
)

model.eval()

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("Model parameters:", f"{model.num_parameters():,}")
print("Model mode:", "Evaluation")


In [ ]:

example = dev[0]

messages = example["messages"][:2]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda:0")

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated = output[0][inputs["input_ids"].shape[1]:]

prediction = tokenizer.decode(
    generated,
    skip_special_tokens=True
).strip()

print("Example ID:", example["id"])
print("BASELINE PREDICTION:")
print(prediction)


In [ ]:

try:
    parsed = json.loads(prediction)

    valid_keys = set(parsed.keys()) == set(KEYS)

    print("Valid JSON:", True)
    print("Correct eight keys:", valid_keys)
    print("Generated keys:", list(parsed.keys()))

except json.JSONDecodeError:
    print("Valid JSON:", False)
    print("Correct eight keys:", False)
    print("Reason: Output is not raw JSON")


In [ ]:

import time

model.eval()
baseline_predictions = []
start_time = time.time()

for example in tqdm(dev, desc="Zero-shot baseline"):
    messages = example["messages"][:2]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda:0")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    baseline_predictions.append({
        "id": example["id"],
        "prediction": prediction
    })

    del inputs, output, generated

baseline_path = OUT / "baseline_zero_shot.jsonl"

with baseline_path.open("w", encoding="utf-8") as f:
    for item in baseline_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Predictions generated:", len(baseline_predictions))
print("Saved:", baseline_path)
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))


In [ ]:

with open(PATHS["score.py"], "r", encoding="utf-8") as f:
    print(f.read())


In [ ]:

import subprocess

scoring_path = OUT / "baseline_zero_shot_scoring.jsonl"

with open(scoring_path, "w", encoding="utf-8") as f:
    for item in baseline_predictions:
        f.write(json.dumps({
            "id": item["id"],
            "output": item["prediction"]
        }, ensure_ascii=False) + "\n")

result = subprocess.run(
    [
        sys.executable,
        PATHS["score.py"],
        "--gold", PATHS["dev.jsonl"],
        "--pred", str(scoring_path)
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


In [ ]:

for item in baseline_predictions[:3]:
    print("ID:", item["id"])
    print("RAW OUTPUT:", repr(item["prediction"]))
    print("-" * 60)


In [ ]:

baseline_summary = {
    "model": BASE_ID,
    "method": "zero_shot",
    "dev_examples": 200,
    "json_validity": 0.0,
    "exact_match": 0.0,
    "mean_field_accuracy": 0.0,
    "elapsed_minutes": 20.94,
    "notes": "Raw outputs frequently contain Markdown fences and incorrect schema fields. No output repair was applied."
}

summary_path = OUT / "baseline_zero_shot_metrics.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(baseline_summary, f, indent=2)

print("Saved:", summary_path)
print("Zero-shot baseline documented successfully.")


In [ ]:

BASELINE_2_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print("Baseline 1:", BASE_ID)
print("Baseline 2:", BASELINE_2_ID)
print("Ready to load second baseline.")


In [ ]:

baseline_2_tokenizer = AutoTokenizer.from_pretrained(
    BASELINE_2_ID,
    trust_remote_code=False
)

baseline_2_model = AutoModelForCausalLM.from_pretrained(
    BASELINE_2_ID,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16,
    trust_remote_code=False
)

baseline_2_model.eval()

print("Second baseline loaded successfully!")
print("Model:", BASELINE_2_ID)
print("Parameters:", sum(p.numel() for p in baseline_2_model.parameters()))


In [ ]:

example = dev[0]

prompt = baseline_2_tokenizer.apply_chat_template(
    example["messages"][:2],
    tokenize=False,
    add_generation_prompt=True
)

inputs = baseline_2_tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda:0")

with torch.inference_mode():
    output = baseline_2_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=baseline_2_tokenizer.eos_token_id
    )

generated = output[0][inputs["input_ids"].shape[1]:]

prediction_2 = baseline_2_tokenizer.decode(
    generated,
    skip_special_tokens=True
).strip()

print("ID:", example["id"])
print("Raw prediction:")
print(prediction_2)


In [ ]:

baseline_2_predictions = []
start_time_2 = time.time()

baseline_2_model.eval()

for example in tqdm(dev, desc="Second baseline"):
    prompt = baseline_2_tokenizer.apply_chat_template(
        example["messages"][:2],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = baseline_2_tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda:0")

    with torch.inference_mode():
        output = baseline_2_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=baseline_2_tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    prediction = baseline_2_tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    baseline_2_predictions.append({
        "id": example["id"],
        "output": prediction
    })

    del inputs, output, generated

baseline_2_path = OUT / "baseline_2_predictions.jsonl"

with open(baseline_2_path, "w", encoding="utf-8") as f:
    for item in baseline_2_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Predictions generated:", len(baseline_2_predictions))
print("Saved:", baseline_2_path)
print("Elapsed minutes:", round((time.time() - start_time_2) / 60, 2))


In [ ]:

import subprocess

result = subprocess.run(
    [
        sys.executable,
        PATHS["score.py"],
        "--gold", PATHS["dev.jsonl"],
        "--pred", str(baseline_2_path)
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)


In [ ]:

baseline_2_summary = {
    "model": BASELINE_2_ID,
    "method": "zero_shot",
    "dev_examples": 200,
    "json_validity": 0.16,
    "exact_match": 0.0,
    "mean_field_accuracy": 0.0,
    "elapsed_minutes": 20.36,
    "notes": "Raw greedy outputs scored without JSON repair."
}

summary_path_2 = OUT / "baseline_2_metrics.json"

with open(summary_path_2, "w", encoding="utf-8") as f:
    json.dump(baseline_2_summary, f, indent=2)

print("Saved:", summary_path_2)
print("Both baselines evaluated and documented.")


In [ ]:
tokenizer=AutoTokenizer.from_pretrained(BASE_ID,trust_remote_code=False)
if tokenizer.pad_token is None:tokenizer.pad_token=tokenizer.eos_token

def prompt_ids(messages):return tokenizer.apply_chat_template(messages[:2],tokenize=True,add_generation_prompt=True)
def full_ids(messages):return tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=False)
lengths=np.array([len(full_ids(r['messages'])) for r in clean])
print('Length percentiles:',{f'p{x}':float(np.percentile(lengths,x)) for x in [50,90,95,99,100]})
MAX_LENGTH=int(min(768,max(384,64*np.ceil(np.percentile(lengths,99)/64))))
print('MAX_LENGTH:',MAX_LENGTH,'percent truncated:',round(100*np.mean(lengths>MAX_LENGTH),2))

def encode_example(row):
    msgs=row['messages'];inp=full_ids(msgs);prefix=prompt_ids(msgs)
    if inp[:len(prefix)]!=prefix:
        raise ValueError('Chat template prefix mismatch: inspect the tokenizer template')
    labels=[-100]*len(prefix)+inp[len(prefix):]
    if len(inp)>MAX_LENGTH:
        # Retain the completion and most recent prompt tokens, never truncate completion first.
        inp=inp[-MAX_LENGTH:];labels=labels[-MAX_LENGTH:]
    return {'input_ids':inp,'labels':labels}

def collate(examples):
    pad=tokenizer.pad_token_id
    m=max(len(x['input_ids']) for x in examples)
    ids=[x['input_ids']+[pad]*(m-len(x['input_ids'])) for x in examples]
    att=[[1]*len(x['input_ids'])+[0]*(m-len(x['input_ids'])) for x in examples]
    lbs=[x['labels']+[-100]*(m-len(x['labels'])) for x in examples]
    return {'input_ids':torch.tensor(ids,device='cuda:0'),'attention_mask':torch.tensor(att,device='cuda:0'),'labels':torch.tensor(lbs,device='cuda:0')}
print('Example supervised tokens:',sum(v!=-100 for v in encode_example(clean[0])['labels']))

In [ ]:

truncated_rows = [
    row for row in clean
    if len(full_ids(row["messages"])) > MAX_LENGTH
]

empty_labels = [
    row["id"] for row in clean
    if all(v == -100 for v in encode_example(row)["labels"])
]

print("Total training examples:", len(clean))
print("Truncated examples:", len(truncated_rows))
print("Examples with no supervised tokens:", len(empty_labels))
print("Example truncated IDs:", [r["id"] for r in truncated_rows[:5]])


In [ ]:

affected = []

for row in truncated_rows:
    original = full_ids(row["messages"])
    prefix = prompt_ids(row["messages"])

    removed_tokens = len(original) - MAX_LENGTH

    if removed_tokens > 0:
        affected.append({
            "id": row["id"],
            "removed_tokens": removed_tokens,
            "prompt_tokens": len(prefix),
            "answer_tokens": len(original) - len(prefix)
        })

print("Truncated examples:", len(affected))
print("Examples losing prompt tokens:",
      sum(x["removed_tokens"] <= x["prompt_tokens"] for x in affected))
print("Examples losing answer tokens:",
      sum(x["removed_tokens"] > x["prompt_tokens"] for x in affected))
print("First 5:", affected[:5])


In [ ]:

train_ready = [
    row for row in clean
    if len(full_ids(row["messages"])) <= MAX_LENGTH
]

print("Original cleaned examples:", len(clean))
print("Training-ready examples:", len(train_ready))
print("Excluded long examples:", len(clean) - len(train_ready))

assert len(train_ready) == 1380
assert all(
    len(full_ids(row["messages"])) <= MAX_LENGTH
    for row in train_ready
)

print("Training data verified successfully!")


In [ ]:

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

total_params = sum(p.numel() for p in model.parameters())

print("Trainable parameters:", f"{trainable_params:,}")
print("Total parameters:", f"{total_params:,}")
print("Trainable percentage:", round(100 * trainable_params / total_params, 4), "%")

assert trainable_params < 50_000_000
print("LoRA parameter limit verified successfully!")


In [ ]:

from torch.utils.data import Dataset

class TriageDataset(Dataset):
    def __init__(self, rows):
        self.examples = [encode_example(row) for row in rows]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]

train_dataset = TriageDataset(train_ready)

print("Training examples:", len(train_dataset))
print("First example tokens:", len(train_dataset[0]["input_ids"]))
print("First example supervised tokens:",
      sum(x != -100 for x in train_dataset[0]["labels"]))

assert len(train_dataset) == 1380

print("Training dataset ready!")


In [ ]:

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(OUT / "triage_lora_checkpoints"),
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    seed=42,
    data_seed=42,
    remove_unused_columns=False,
    dataloader_num_workers=0
)

print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Effective batch size:",
      training_args.per_device_train_batch_size *
      training_args.gradient_accumulation_steps)
print("Learning rate:", training_args.learning_rate)
print("Training configuration ready!")


In [ ]:

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate
)

print("Trainer initialized successfully!")
print("Training examples:", len(trainer.train_dataset))
print("Training epochs:", training_args.num_train_epochs)
print("Ready to start QLoRA training.")


In [ ]:

import time

training_start = time.time()

train_result = trainer.train()

training_minutes = (time.time() - training_start) / 60

print("Training completed successfully!")
print("Training time (minutes):", round(training_minutes, 2))
print("Final training loss:", train_result.training_loss)
print("Global steps:", trainer.state.global_step)


In [ ]:

# Fix: Create batch tensors on CPU.
# Trainer will automatically transfer them to the GPU.

def collate(examples):
    pad = tokenizer.pad_token_id
    m = max(len(x["input_ids"]) for x in examples)

    ids = [
        x["input_ids"] + [pad] * (m - len(x["input_ids"]))
        for x in examples
    ]

    att = [
        [1] * len(x["input_ids"]) +
        [0] * (m - len(x["input_ids"]))
        for x in examples
    ]

    lbs = [
        x["labels"] + [-100] * (m - len(x["labels"]))
        for x in examples
    ]

    return {
        "input_ids": torch.tensor(ids, dtype=torch.long),
        "attention_mask": torch.tensor(att, dtype=torch.long),
        "labels": torch.tensor(lbs, dtype=torch.long)
    }

# Update the existing Trainer
trainer.data_collator = collate

# Verify the fix
sample_batch = collate([train_dataset[0], train_dataset[1]])

print("Batch device:", sample_batch["input_ids"].device)
print("Batch shape:", sample_batch["input_ids"].shape)
print("Collate function fixed successfully!")


In [ ]:

import time

training_start = time.time()

train_result = trainer.train()

training_minutes = (time.time() - training_start) / 60

print("Training completed successfully!")
print("Training time (minutes):", round(training_minutes, 2))
print("Final training loss:", train_result.training_loss)
print("Global steps:", trainer.state.global_step)


In [ ]:

import torch

print("Visible GPU count:", torch.cuda.device_count())
print("Trainer GPU count:", training_args.n_gpu)
print("Trainer device:", training_args.device)
print("Model device:", next(model.parameters()).device)
print("Trainer model wrapped:", type(trainer.model_wrapped).__name__)


In [ ]:

# Use only GPU 0 for QLoRA training
training_args._n_gpu = 1

# Clear any gradients left by failed training attempts
model.zero_grad(set_to_none=True)

print("Trainer GPU count:", training_args.n_gpu)
print("Training device:", training_args.device)
print("Model device:", next(model.parameters()).device)

assert training_args.n_gpu == 1
print("Single-GPU configuration verified!")


In [ ]:

import time

training_start = time.time()

train_result = trainer.train()

training_minutes = (time.time() - training_start) / 60

print("Training completed successfully!")
print("Training time (minutes):", round(training_minutes, 2))
print("Final training loss:", train_result.training_loss)
print("Global steps:", trainer.state.global_step)



## C. Loading, fine-tuning and faithful inference

We use 4-bit NF4 quantization with LoRA rank 8 for the reference. The FP16 comparison loads the *same* 1.5B model without quantization. Only attention projections `q_proj` and `v_proj` are adapted, keeping trainable parameters far below 50M.

For the required 16-bit LoRA comparison, weights are loaded in `torch.float16` (not 4-bit). Both methods are compatible with T4.

In [ ]:
def load_base(quantized):
    kw={'torch_dtype':torch.float16,'device_map':{'':'cuda:0'},'trust_remote_code':False}
    if quantized:
        kw['quantization_config']=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
    model=AutoModelForCausalLM.from_pretrained(BASE_ID,**kw)
    model.config.use_cache=False
    if quantized:model=prepare_model_for_kbit_training(model,use_gradient_checkpointing=True)
    else:
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
    return model

def attach_lora(base,rank):
    config=LoraConfig(r=rank,lora_alpha=rank*2,lora_dropout=0.05,target_modules=['q_proj','v_proj'],bias='none',task_type='CAUSAL_LM')
    model=get_peft_model(base,config)
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    print('Trainable parameters:',f'{trainable:,}')
    assert trainable<=50_000_000
    return model

def infer_rows(model,rows,system_override=None,desc='inference'):
    model.eval();pred=[]
    for row in tqdm(rows,desc=desc):
        messages=copy.deepcopy(row['messages'][:2])
        if system_override is not None:messages[0]['content']=system_override
        ids=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda:0')
        with torch.inference_mode():
            answer=model.generate(input_ids=ids,do_sample=False,max_new_tokens=200,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
        output=tokenizer.decode(answer[0,ids.shape[1]:],skip_special_tokens=True).strip()
        pred.append({'id':row['id'],'output':output})
    return pred

def write_preds(path,rows):
    with open(path,'w',encoding='utf8') as f:
        for r in rows:f.write(json.dumps(r,ensure_ascii=False)+'\n')

spec=importlib.util.spec_from_file_location('assignment_score',PATHS['score.py'])
scorer=importlib.util.module_from_spec(spec);spec.loader.exec_module(scorer)
def evaluate(predictions):return scorer.score(dev,predictions)

def release(model):
    del model;gc.collect();torch.cuda.empty_cache()


## D. Required baselines on all 200 dev rows

Only for the rules-in-prompt reference baseline, append the provided schema text to the fixed system instruction. This is **never** used for fine-tuned model inference.

In [ ]:
base=load_base(quantized=True)
base.config.use_cache=True
baseline_fixed=infer_rows(base,dev,desc='baseline fixed system')
write_preds(OUT/'baseline_fixed.jsonl',baseline_fixed)
fixed_res=evaluate(baseline_fixed)
RULES=pathlib.Path(PATHS['SCHEMA.md']).read_text(encoding='utf8')
original_system=dev[0]['messages'][0]['content']
with_rules=original_system+'\n\nFollow this specification exactly:\n'+RULES
baseline_rules=infer_rows(base,dev,system_override=with_rules,desc='baseline with rules')
write_preds(OUT/'baseline_rules.jsonl',baseline_rules)
rules_res=evaluate(baseline_rules)
print('Baseline fixed:',fixed_res['all'])
print('Baseline rules:',rules_res['all'])
release(base);del base

## E. Five controlled runs (four one-variable comparisons)

Reference: cleaned data / 4-bit QLoRA / rank 8 / 1 epoch.

1. Change only **data** to raw. 2. Change only **precision** to 16-bit LoRA. 3. Change only **rank** to 16. 4. Change only **epochs** to 2. Same seed, learning rate, batch sizes, target modules, and dev evaluation for all.

**Practical note:** Executing all five may consume significant Kaggle GPU quota. Save outputs between runs. Train each from the original base model, not from the previous run.

In [ ]:
EXPERIMENTS=[
 {'name':'clean_qlora_r8_e1','cleaned':True,'quantized':True,'rank':8,'epochs':1},
 {'name':'raw_qlora_r8_e1','cleaned':False,'quantized':True,'rank':8,'epochs':1},
 {'name':'clean_lora16_r8_e1','cleaned':True,'quantized':False,'rank':8,'epochs':1},
 {'name':'clean_qlora_r16_e1','cleaned':True,'quantized':True,'rank':16,'epochs':1},
 {'name':'clean_qlora_r8_e2','cleaned':True,'quantized':True,'rank':8,'epochs':2},
]
LR=2e-4; GRAD_ACCUM=8

def train_run(cfg):
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
    torch.cuda.reset_peak_memory_stats(0)
    model=attach_lora(load_base(cfg['quantized']),cfg['rank'])
    model.train()
    dataset=clean if cfg['cleaned'] else raw
    items=[encode_example(r) for r in dataset]
    # Assistant-only supervised loss. Every example includes EOS/end-of-turn.
    assert all(any(k!=-100 for k in x['labels']) for x in items)
    optim=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LR)
    scaler=torch.amp.GradScaler('cuda',enabled=not cfg['quantized'])
    tic=time.perf_counter();steps=0
    for epoch in range(cfg['epochs']):
        order=list(range(len(items)));random.Random(SEED+epoch).shuffle(order)
        optim.zero_grad(set_to_none=True)
        for j,idx in enumerate(tqdm(order,desc=cfg['name']+f' epoch {epoch+1}')):
            batch=collate([items[idx]])
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                loss=model(**batch).loss/GRAD_ACCUM
            scaler.scale(loss).backward()
            if (j+1)%GRAD_ACCUM==0 or j+1==len(order):
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(optim);scaler.update();optim.zero_grad(set_to_none=True);steps+=1
    minutes=(time.perf_counter()-tic)/60
    peak_gb=torch.cuda.max_memory_allocated(0)/(1024**3)
    model.config.use_cache=True
    preds=infer_rows(model,dev,desc='dev '+cfg['name'])
    write_preds(OUT/(cfg['name']+'_dev.jsonl'),preds)
    metrics=evaluate(preds)['all']
    path=OUT/'candidate_adapters'/cfg['name']
    path.mkdir(parents=True,exist_ok=True)
    model.save_pretrained(str(path),safe_serialization=True)
    result={**cfg,'dev_exact_match':metrics['exact_match'],'dev_mean_field_accuracy':metrics['mean_field_acc'],'peak_gpu_gb':round(peak_gb,3),'train_minutes':round(minutes,2),'optimizer_steps':steps,'adapter_path':str(path)}
    print(json.dumps(result,indent=2))
    del optim,model,items,preds
    gc.collect();torch.cuda.empty_cache()
    return result

results=[]
for cfg in EXPERIMENTS:
    results.append(train_run(cfg))
    with (OUT/'experiments.json').open('w') as f:json.dump(results,f,indent=2)
with (OUT/'experiments.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=results[0].keys());writer.writeheader();writer.writerows(results)
print('Completed experiments:',len(results))

## F. Best adapter, error analysis, and test predictions

Select by dev exact match (then field accuracy) without ever looking at hidden-test labels. Record at least 10 dev mistakes and field-level accuracy in the report. Test outputs are **raw decoded model responses** without corrections.

In [ ]:
best=max(results,key=lambda r:(r['dev_exact_match'],r['dev_mean_field_accuracy']))
print('Chosen best:',best)
final_adapter=OUT/'adapter'
if final_adapter.exists():shutil.rmtree(final_adapter)
shutil.copytree(best['adapter_path'],final_adapter)

best_devs=read_jsonl(OUT/(best['name']+'_dev.jsonl'))
best_metrics=evaluate(best_devs)
print('Dev metrics by field:',json.dumps(best_metrics['all'],indent=2))
print('Dev metrics by category:',{k:round(v['exact_match'],3) for k,v in best_metrics.items() if k.startswith('category=')})
preds_by_id={r['id']:r['output'] for r in best_devs}
failures=[]
for row in dev:
    expected=json.loads(row['messages'][-1]['content'])
    output=preds_by_id[row['id']]
    try:pred=json.loads(output)
    except:pred={}
    wrong=[k for k in scorer.KEYS if k not in pred or not scorer.same(pred[k],expected[k])]
    if set(pred)!=set(scorer.KEYS):wrong.append('JSON keys')
    if wrong:failures.append({'id':row['id'],'ticket':row['messages'][1]['content'],'expected':expected,'predicted_raw':output,'wrong_fields':wrong})
with (OUT/'dev_failures.json').open('w',encoding='utf8') as f:json.dump(failures,f,ensure_ascii=False,indent=2)
print('Failures:',len(failures),'first 10:')
for failure in failures[:10]:print(json.dumps(failure,ensure_ascii=False)[:1100])

finalbase=load_base(best['quantized']);finalmodel=PeftModel.from_pretrained(finalbase,str(final_adapter))
finalmodel.config.use_cache=True
pred_test=infer_rows(finalmodel,test,desc='FINAL hidden test')
write_preds(OUT/'predictions_test.jsonl',pred_test)
assert len(pred_test)==len(test) and all(a['id']==b['id'] for a,b in zip(pred_test,test))
print('Test predictions saved:',OUT/'predictions_test.jsonl')
release(finalmodel)

## G. Report and README drafts (fill in manual insights)

The report contains measured numbers from this notebook and placeholders for your **own** explanations of at least 10 errors. Do not claim to have run experiments you did not run. Review the sampled failures and write explanations before submission.

In [ ]:
lines=['# VIMA3YA triage — Experiment report','',
 '## Data audit',f'- Input training rows: {len(raw)}; retained: {len(clean)}; dropped: {sum(x["action"]=="drop" for x in log)}; repaired: {sum(x["action"]=="fix" for x in log)}.',
 '- See `cleaning_log.csv` for row-level changes. Invalid JSON was dropped. Parseable dates and NP IDs were normalized; priority and forced P1 human escalation were updated according to SCHEMA.md.',
 '- Limitations: text-sensitive fields such as category and previous-contact intent may still be mislabeled. Repairs use training data and schema only.',
 '', '## Baselines',f'- Fixed prompt: exact match {fixed_res["all"]["exact_match"]:.4f}; field accuracy {fixed_res["all"]["mean_field_acc"]:.4f}.',
 f'- Rules prompt (reference only): exact match {rules_res["all"]["exact_match"]:.4f}; field accuracy {rules_res["all"]["mean_field_acc"]:.4f}.',
 '', '## Controlled experiments', '| Run | Dev exact | Mean field | Peak GPU GiB | Train min |','|---|---:|---:|---:|---:|']
for r in results:lines.append(f'| {r["name"]} | {r["dev_exact_match"]:.4f} | {r["dev_mean_field_accuracy"]:.4f} | {r["peak_gpu_gb"]:.2f} | {r["train_minutes"]:.1f} |')
lines+=['',f'Best by dev exact match: **{best["name"]}**. Hyperparameters: LR={LR}, grad accumulation={GRAD_ACCUM}; LoRA targets q_proj and v_proj, dropout 0.05. Training length selected from measured p99, max={MAX_LENGTH}.',
'','## Error analysis',f'Overall field accuracies: {best_metrics["all"]["fields"]}',
'Weakest categories (dev exact match): '+str(sorted([(k,v['exact_match']) for k,v in best_metrics.items() if k.startswith('category=')],key=lambda x:x[1])[:3]),
'','### Ten concrete errors — add explanations manually']
for e in failures[:10]:lines.append(f'- **{e["id"]}**: incorrect {", ".join(e["wrong_fields"])}. **Explanation to complete after reviewing ticket and expected/predicted outputs:** [YOUR ANALYSIS]')
lines+=['','## With one more week','- Manually verify the remaining noisy training rows; add train-only counterexamples for legal threats, relative dates, normalization and confusing balances; assess multiple random seeds and calibration without using dev as training.','','## AI assistance','- ChatGPT assisted with initial code, notebook structure, audit proposals and report scaffold; candidate reviewed, ran and verified all code, numbers and explanations (edit this sentence to reflect what you actually did).','- Training and predictions were produced locally using the stated open model and PEFT stack; no external model/API generated labels or test outputs.']
(OUT/'report.md').write_text('\n'.join(lines)+'\n',encoding='utf8')
(OUT/'README.md').write_text(f"""# VIMA3YA NimbusPay Ticket Triage

Base model: `{BASE_ID}`. Best run: `{best['name']}`; 4-bit base loading: `{best['quantized']}`. Dev exact match {best['dev_exact_match']:.4f}; mean field accuracy {best['dev_mean_field_accuracy']:.4f}.

## Contents
- `notebook.ipynb`: training, controlled experiments, inference.
- `adapter/`: PEFT LoRA adapter.
- `predictions_test.jsonl`: model-generated predictions.
- `cleaning_log.csv`: documented training repairs and drops.
- `report.md`: audit, baselines, experiments, error analysis and AI-use disclosure.

To reproduce, upload the provided train/dev/test JSONL, schema and score script to the notebook as described at the top. Data and base weights are not committed to GitHub.
""",encoding='utf8')
print('Generated README.md and report.md. Review report before committing.')

## H. Gather submission files

Do **not** put raw training/dev/test files or the base model into your public/private GitHub repo. Copy these outputs to your repo after reviewing them: `adapter/`, `predictions_test.jsonl`, `cleaning_log.csv`, `README.md`, `report.md`, and this notebook **with run outputs**. Kaggle's **Save Version → Save & Run All** can preserve notebook outputs; then download the executed `.ipynb`. Keep your code and results committed **before the deadline**.

In [ ]:
print('Required outputs:')
for filename in ['README.md','report.md','cleaning_log.csv','predictions_test.jsonl','experiments.csv']:
    file=OUT/filename;print(filename,file.exists(),file.stat().st_size if file.exists() else None)
print('Adapter files:',[x.name for x in (OUT/'adapter').glob('*')])

In [ ]:

import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))


In [ ]:

import json
from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/keerthiyp/assignment")

for filename in [
    "train.jsonl",
    "dev.jsonl",
    "test_inputs.jsonl"
]:
    path = DATA_DIR / filename

    with open(path, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    print(f"{filename}: {len(rows)} records")


In [ ]:
/kaggle/input/datasets/keerthiyp/assignment

In [ ]:
from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/keerthiyp/assignment")

print(DATA_DIR.exists())

In [ ]:

import json
from pathlib import Path

raw_adapter_dir = Path("/kaggle/working/experiment1_raw_adapter")
raw_adapter_dir.mkdir(parents=True, exist_ok=True)

# Save trained LoRA adapter
raw_trainer.save_model(str(raw_adapter_dir))
tokenizer.save_pretrained(str(raw_adapter_dir))

# Save measured training statistics
raw_training_stats = {
    "experiment": "raw_vs_cleaned_raw",
    "training_examples": len(raw_train_dataset),
    "epochs": 2,
    "global_steps": raw_trainer.state.global_step,
    "training_minutes": raw_training_minutes,
    "training_loss": raw_train_result.training_loss,
    "peak_gpu_memory_gb": raw_peak_gpu_gb,
    "trainable_parameters": 18464768
}

stats_path = Path(
    "/kaggle/working/experiment1_raw_training_stats.json"
)

with open(stats_path, "w") as f:
    json.dump(raw_training_stats, f, indent=2)

print("Raw-data adapter saved:", raw_adapter_dir)
print("Training statistics saved:", stats_path)
print("\nAdapter files:")
for file in sorted(raw_adapter_dir.iterdir()):
    print(" -", file.name, round(file.stat().st_size / 1024**2, 2), "MB")

assert (raw_adapter_dir / "adapter_model.safetensors").exists()
assert stats_path.exists()

print("\nExperiment 1 raw-data model safely saved!")


In [ ]:

import json
import time
from pathlib import Path
from tqdm.auto import tqdm

raw_model.eval()

raw_predictions = []
eval_start = time.time()

for row in tqdm(dev_rows, desc="Raw model dev evaluation"):
    inputs = tokenizer.apply_chat_template(
        row["messages"][:2],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda:0")

    with torch.inference_mode():
        generated = raw_model.generate(
            input_ids=inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    raw_output = tokenizer.decode(
        generated[0][inputs.shape[1]:],
        skip_special_tokens=True
    ).strip()

    raw_predictions.append({
        "id": row["id"],
        "output": raw_output
    })

raw_eval_minutes = (time.time() - eval_start) / 60

raw_predictions_path = Path(
    "/kaggle/working/experiment1_raw_dev_predictions.jsonl"
)

with open(raw_predictions_path, "w", encoding="utf-8") as f:
    for prediction in raw_predictions:
        f.write(json.dumps(prediction, ensure_ascii=False) + "\n")

print("\n===== RAW MODEL EVALUATION =====")
print("Predictions saved:", raw_predictions_path)
print("Total predictions:", len(raw_predictions))
print("Evaluation time:", round(raw_eval_minutes, 2), "minutes")
print("First raw output:")
print(raw_predictions[0]["output"])

assert len(raw_predictions) == len(dev_rows) == 200
print("\nEvaluation completed successfully!")


In [ ]:

import shutil
from IPython.display import FileLink, display

shutil.make_archive(
    "/kaggle/working/experiment1_raw_backup",
    "zip",
    "/kaggle/working",
    "experiment1_raw_adapter"
)

display(FileLink("experiment1_raw_backup.zip"))


In [ ]:

from pathlib import Path
import torch

paths = {
    "Raw adapter": "/kaggle/working/experiment1_raw_adapter/adapter_model.safetensors",
    "Raw training stats": "/kaggle/working/experiment1_raw_training_stats.json",
    "Cleaned adapter": "/kaggle/working/adapter/adapter_model.safetensors",
    "Dev dataset": "/kaggle/input/datasets/keerthiyp/assignment/dev.jsonl"
}

for name, path in paths.items():
    print(name, ":", "FOUND" if Path(path).exists() else "MISSING")

print("GPU available:", torch.cuda.is_available())
print("Raw model in memory:", "raw_model" in globals())
print("Tokenizer in memory:", "tokenizer" in globals())
print("Dev rows in memory:", "dev_rows" in globals())


In [ ]:

from pathlib import Path

print("Backup ZIP files found:")

for file in Path("/kaggle/input").rglob("*.zip"):
    print(file)


In [ ]:

from pathlib import Path

print("Files available in Kaggle Input:\n")

for file in Path("/kaggle/input").rglob("*"):
    if file.is_file():
        print(file)


In [ ]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

BASE_ID = "Qwen/Qwen2.5-1.5B-Instruct"

RAW_ADAPTER = (
    "/kaggle/input/datasets/keerthiyp/assignment2/"
    "experiment1_raw_backup/experiment1_raw_adapter"
)

tokenizer = AutoTokenizer.from_pretrained(RAW_ADAPTER)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16,
    trust_remote_code=False
)

raw_model = PeftModel.from_pretrained(
    base_model,
    RAW_ADAPTER
)

raw_model.eval()

print("Raw-data model restored successfully!")
print("Device:", next(raw_model.parameters()).device)
print("Visible GPUs:", torch.cuda.device_count())


In [ ]:

%pip install -q -U "bitsandbytes>=0.46.1" "accelerate>=1.0.0" "peft>=0.13.0"

import importlib.metadata as metadata

print("bitsandbytes:", metadata.version("bitsandbytes"))
print("accelerate:", metadata.version("accelerate"))
print("peft:", metadata.version("peft"))

print("\nInstallation completed!")


In [1]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import bitsandbytes
import peft
import transformers

print("GPU available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())
print("bitsandbytes:", bitsandbytes.__version__)
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 1

print("Environment ready for model recovery!")


GPU available: True
Visible GPUs: 1
bitsandbytes: 0.50.2
PEFT: 0.21.2
Transformers: 5.20.2
Environment ready for model recovery!


In [2]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

BASE_ID = "Qwen/Qwen2.5-1.5B-Instruct"

RAW_ADAPTER = (
    "/kaggle/input/datasets/keerthiyp/assignment2/"
    "experiment1_raw_backup/experiment1_raw_adapter"
)

tokenizer = AutoTokenizer.from_pretrained(RAW_ADAPTER)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16
)

raw_model = PeftModel.from_pretrained(
    base_model,
    RAW_ADAPTER
)

raw_model.eval()

print("Raw-data model restored successfully!")
print("Device:", next(raw_model.parameters()).device)


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Raw-data model restored successfully!
Device: cuda:0


In [4]:

import json
import time
import torch
from pathlib import Path

# Load development dataset
dev_path = Path(
    "/kaggle/input/datasets/keerthiyp/assignment/dev.jsonl"
)

with open(dev_path, "r", encoding="utf-8") as f:
    dev_rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Development examples:", len(dev_rows))

# Prepare output file
output_path = Path(
    "/kaggle/working/experiment1_raw_dev_predictions.jsonl"
)

raw_predictions = []

raw_model.eval()
start = time.time()

# Evaluate all 200 examples
for i, row in enumerate(dev_rows):

    # Use original system and user messages only
    inputs = tokenizer.apply_chat_template(
        row["messages"][:2],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda:0")

    # Greedy inference
    with torch.inference_mode():
        generated = raw_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode generated tokens only
    output = tokenizer.decode(
        generated[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    raw_predictions.append({
        "id": row["id"],
        "output": output
    })

    # Save progress every 25 examples
    if (i + 1) % 25 == 0:
        with open(output_path, "w", encoding="utf-8") as f:
            for prediction in raw_predictions:
                f.write(
                    json.dumps(prediction, ensure_ascii=False)
                    + "\n"
                )

        print(f"Completed {i + 1}/{len(dev_rows)}")

# Final save
with open(output_path, "w", encoding="utf-8") as f:
    for prediction in raw_predictions:
        f.write(
            json.dumps(prediction, ensure_ascii=False)
            + "\n"
        )

print("\n===== EXPERIMENT 1 EVALUATION =====")
print("Evaluation completed successfully!")
print("Total predictions:", len(raw_predictions))
print(
    "Time (minutes):",
    round((time.time() - start) / 60, 2)
)
print("Saved:", output_path)


Development examples: 200
Completed 25/200
Completed 50/200
Completed 75/200
Completed 100/200
Completed 125/200
Completed 150/200
Completed 175/200
Completed 200/200

===== EXPERIMENT 1 EVALUATION =====
Evaluation completed successfully!
Total predictions: 200
Time (minutes): 21.2
Saved: /kaggle/working/experiment1_raw_dev_predictions.jsonl


In [5]:

import subprocess
import json
from pathlib import Path

result = subprocess.run(
    [
        "python",
        "/kaggle/input/datasets/keerthiyp/assignment/score.py",
        "--gold",
        "/kaggle/input/datasets/keerthiyp/assignment/dev.jsonl",
        "--pred",
        "/kaggle/working/experiment1_raw_dev_predictions.jsonl",
        "--json"
    ],
    capture_output=True,
    text=True,
    check=True
)

raw_scores = json.loads(result.stdout)

metrics_path = Path(
    "/kaggle/working/experiment1_raw_dev_metrics.json"
)

metrics_path.write_text(
    json.dumps(raw_scores, indent=2),
    encoding="utf-8"
)

print("===== EXPERIMENT 1: RAW DATA =====")
print("JSON validity:", f"{raw_scores['all']['json_valid']:.2%}")
print("Exact match:", f"{raw_scores['all']['exact_match']:.2%}")
print("Mean field accuracy:", f"{raw_scores['all']['mean_field_acc']:.2%}")

print("\n===== PER-FIELD ACCURACY =====")
for field, accuracy in raw_scores["all"]["fields"].items():
    print(f"{field}: {accuracy:.2%}")

print("\nMetrics saved:", metrics_path)


===== EXPERIMENT 1: RAW DATA =====
JSON validity: 100.00%
Exact match: 57.00%
Mean field accuracy: 92.56%

===== PER-FIELD ACCURACY =====
category: 97.50%
priority: 61.50%
amount_paise: 96.50%
txn_date: 98.00%
txn_id: 100.00%
channel: 99.50%
language: 100.00%
needs_human: 87.50%

Metrics saved: /kaggle/working/experiment1_raw_dev_metrics.json


In [6]:

import zipfile
from pathlib import Path
from IPython.display import FileLink, display

working = Path("/kaggle/working")

files = [
    working / "experiment1_raw_dev_predictions.jsonl",
    working / "experiment1_raw_dev_metrics.json"
]

backup = working / "experiment1_results_backup.zip"

with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files:
        assert file.exists(), f"Missing: {file}"
        z.write(file, arcname=file.name)

print("Experiment 1 results backed up!")
print("Backup size:", round(backup.stat().st_size / 1024, 2), "KB")

display(FileLink(backup.name))



Experiment 1 results backed up!
Backup size: 3.95 KB


/kaggle/working/experiment1_results_backup.zip

In [7]:

import json
from pathlib import Path

test_path = Path(
    "/kaggle/input/datasets/keerthiyp/assignment/test_inputs.jsonl"
)

with open(test_path, encoding="utf-8") as f:
    test_rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Test examples:", len(test_rows))
print("First test ID:", test_rows[0]["id"])
print("Available keys:", list(test_rows[0].keys()))

assert len(test_rows) == 400

print("Test dataset ready!")


Test examples: 400
First test ID: ts-00001
Available keys: ['id', 'messages']
Test dataset ready!


In [8]:

import json
import time
import torch
from pathlib import Path

output_path = Path("/kaggle/working/predictions_test.jsonl")

test_predictions = []
raw_model.eval()
start = time.time()

for i, row in enumerate(test_rows):

    # Original system and user messages only
    inputs = tokenizer.apply_chat_template(
        row["messages"][:2],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda:0")

    # Required greedy decoding
    with torch.inference_mode():
        generated = raw_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    # Raw model output, whitespace stripped only
    output = tokenizer.decode(
        generated[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    test_predictions.append({
        "id": row["id"],
        "output": output
    })

    # Save progress every 25 examples
    if (i + 1) % 25 == 0:
        with open(output_path, "w", encoding="utf-8") as f:
            for prediction in test_predictions:
                f.write(json.dumps(prediction, ensure_ascii=False) + "\n")

        print(
            f"Completed {i + 1}/{len(test_rows)}",
            flush=True
        )

# Final save
with open(output_path, "w", encoding="utf-8") as f:
    for prediction in test_predictions:
        f.write(json.dumps(prediction, ensure_ascii=False) + "\n")

print("\n===== FINAL TEST GENERATION =====")
print("Total predictions:", len(test_predictions))
print("Time:", round((time.time() - start) / 60, 2), "minutes")
print("Saved:", output_path)

assert len(test_predictions) == 400
print("Final test predictions generated successfully!")


Completed 25/400
Completed 50/400
Completed 75/400
Completed 100/400
Completed 125/400
Completed 150/400
Completed 175/400
Completed 200/400
Completed 225/400
Completed 250/400
Completed 275/400
Completed 300/400
Completed 325/400
Completed 350/400
Completed 375/400
Completed 400/400

===== FINAL TEST GENERATION =====
Total predictions: 400
Time: 40.89 minutes
Saved: /kaggle/working/predictions_test.jsonl
Final test predictions generated successfully!


In [9]:

import json
from pathlib import Path
from IPython.display import FileLink, display

path = Path("/kaggle/working/predictions_test.jsonl")

with open(path, encoding="utf-8") as f:
    predictions = [json.loads(line) for line in f if line.strip()]

assert len(predictions) == 400, "Expected 400 predictions"
assert len({p["id"] for p in predictions}) == 400, "Duplicate IDs found"
assert [p["id"] for p in predictions] == [r["id"] for r in test_rows], "ID order mismatch"

valid_json = 0
for p in predictions:
    try:
        json.loads(p["output"])
        valid_json += 1
    except (json.JSONDecodeError, TypeError):
        pass

print("Total predictions:", len(predictions))
print("Unique IDs:", len({p["id"] for p in predictions}))
print("Valid output JSON:", valid_json, "/ 400")
print("File size:", round(path.stat().st_size / 1024, 2), "KB")

display(FileLink(str(path)))


Total predictions: 400
Unique IDs: 400
Valid output JSON: 400 / 400
File size: 87.51 KB


/kaggle/working/predictions_test.jsonl

In [10]:

import zipfile
from pathlib import Path
from IPython.display import FileLink, display

source = Path("/kaggle/working/predictions_test.jsonl")
zip_path = Path("/kaggle/working/final_test_predictions.zip")

assert source.exists(), "Prediction file not found!"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(source, arcname="predictions_test.jsonl")

print("ZIP created successfully!")
print("Size:", round(zip_path.stat().st_size / 1024, 2), "KB")

display(FileLink(str(zip_path)))


ZIP created successfully!
Size: 5.9 KB


/kaggle/working/final_test_predictions.zip